# 02 — Reduction Kernels

Covers Phase 2 of the kernel roadmap:
- `reduce_sum` — parallel tree reduction
- `max_min` — argmax, argmin
- `softmax` — row-wise, numerically stable
- `layer_norm` — forward pass

**Metric**: GB/s for most; softmax/layer_norm use `2 × n × bytes` (read once + write once).

In [ ]:
# ── Setup: mount Drive and clone / pull the repo ─────────────────────────────
import os
from google.colab import drive

drive.mount("/content/drive")

REPO_URL    = "https://github.com/Bhavikupadhyay/triton-kernels.git"
REPO_BRANCH = "feature/max-min"
REPO_DIR    = "/content/drive/MyDrive/triton-kernels"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --all
    !git -C {REPO_DIR} checkout -f {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git rev-parse --abbrev-ref HEAD
!bash scripts/setup_colab.sh

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch
import triton

from kernels.reductions.reduce_sum import reduce_sum, test_reduce_sum, benchmark_reduce_sum
from kernels.reductions.max_min import argmax, argmin, test_max_min, benchmark_argmax, benchmark_argmin

# Uncomment as kernels are implemented:
# from kernels.reductions.softmax import softmax, test_softmax, benchmark_softmax
# from kernels.reductions.layer_norm import layer_norm, test_layer_norm, benchmark_layer_norm

print("Imports ready")

## 1. reduce_sum

**File**: `kernels/reductions/reduce_sum.py`  
**PyTorch equivalent**: `torch.sum(x)`  
**Design**: two-pass — each block reduces BLOCK_SIZE elements into one partial sum; a second single-block pass collapses the partials into the final scalar  
**Metric**: GB/s — reads 1 tensor → `(1 × n × bytes × 1e-9) / (ms × 1e-3)`

In [ ]:
# ── reduce_sum: Correctness ───────────────────────────────────────────────────
test_reduce_sum()

In [ ]:
# ── reduce_sum: Benchmark ─────────────────────────────────────────────────────
benchmark_reduce_sum.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/reductions",
)

**Interpretation — reduce_sum (T4, fp32)**

**Triton consistently beats PyTorch by ~4–8%** across all sizes. This is a different result from Phase 1 — elementwise kernels matched PyTorch exactly. Reductions are harder to optimise generically, and our focused two-pass design edges out PyTorch's general-purpose CUB reduction path.

**Small n is kernel-launch latency dominated.** At n=4096, only 4 blocks are dispatched — the GPU is barely engaged. The only crossover where PyTorch wins is at n=8192 (3.9 vs 3.2 GB/s): at that size the second kernel launch in our two-pass design costs proportionally more than PyTorch's single-pass path.

**Peak bandwidth is higher than Phase 1 elementwise kernels** (~286 GB/s vs ~241 GB/s). The reason: `reduce_sum` is a read-only workload. The scratch buffer (partials) and the final scalar are tiny — they live in L2/registers. HBM is touched only once (the sequential read of `x` in pass 1). No HBM write means no bandwidth contention between reads and writes, allowing closer saturation of the read bandwidth ceiling.

**Both converge near the T4's practical peak** at large n — Triton at 286 GB/s (~89% of 320 GB/s theoretical), PyTorch at 275 GB/s (~86%). The remaining ~11% gap to theoretical is normal cache and scheduler overhead.

**The two-pass overhead is invisible at large n.** Pass 2 sums at most ~131K fp32 partials (0.5 MB), which fits comfortably in T4's 4 MB L2, adding negligible time.

## 2. max_min

**File**: `kernels/reductions/max_min.py`  
**PyTorch equivalents**: `torch.max(x)` / `torch.argmax(x)`, `torch.min(x)` / `torch.argmin(x)`  
**Design**: same two-pass structure as `reduce_sum` — pass 1 finds the block-local extreme value + index, pass 2 finds the global extreme across partials  
**Metric**: GB/s — reads 1 tensor → `(1 × n × bytes × 1e-9) / (ms × 1e-3)`

In [ ]:
# ── max_min: Correctness ──────────────────────────────────────────────────────
test_max_min()

In [ ]:
# ── max_min: Benchmark ───────────────────────────────────────────────────────
benchmark_argmax.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/reductions",
)
benchmark_argmin.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/reductions",
)

**Interpretation**: Add notes here.

## 3. softmax

**File**: `kernels/reductions/softmax.py`  
**PyTorch equivalent**: `torch.softmax(x, dim=-1)`  
**Key**: numerically stable via max subtraction (online softmax trick)  
**Metric**: GB/s — `(2 × n × bytes × 1e-9) / (ms × 1e-3)` (read + write)

In [ ]:
# ── softmax: Correctness ─────────────────────────────────────────────────────
# test_softmax()

In [ ]:
# ── softmax: Benchmark ───────────────────────────────────────────────────────
# benchmark_softmax.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/reductions/softmax_benchmark.png")

**Interpretation**: Add notes here.

## 4. layer_norm

**File**: `kernels/reductions/layer_norm.py`  
**PyTorch equivalent**: `torch.nn.functional.layer_norm(x, normalized_shape)`  
**Metric**: GB/s — `(2 × n × bytes × 1e-9) / (ms × 1e-3)`

In [ ]:
# ── layer_norm: Correctness ──────────────────────────────────────────────────
# test_layer_norm()

In [ ]:
# ── layer_norm: Benchmark ────────────────────────────────────────────────────
# benchmark_layer_norm.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/reductions/layer_norm_benchmark.png")

**Interpretation**: Add notes here.

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
# import pandas as pd, glob
# csvs = glob.glob("benchmarks/results/reductions/*.csv")
# if csvs:
#     print(pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True).to_string(index=False))
# else:
#     print("No CSVs yet.")